In [ ]:
import pylbsr.notebooks
import pylbsr.misc

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from dotmap import DotMap
from tqdm import tqdm

import parnet
from parnet_additional_utils import GzListDataset, ParnetModelName, load_parnet_model
from parnet.layers import LinearProjectionHead

/home/lhofer/pixi-envs/parnet--unified-7433550039317049837/envs/parnet-dev-cu12/lib/python3.10/site-packages/gin/config.py:615: FutureWarning: `NLLLoss2d` has been deprecated. Please use `NLLLoss` instead as a drop-in replacement and see https://pytorch.org/docs/main/nn.html#torch.nn.NLLLoss for more details.
  decorated_class = decorating_meta(cls.__name__, (cls,), overrides)
Seed set to 42


In [2]:
_notebook_name = "03_integrated_gradients.ipynb"
_notebook_path = f"notebooks/globalclip-head/{_notebook_name}"

pylbsr.notebooks.enable_cell_timing_metadata(show=True)
logger = pylbsr.misc.init_logger(_notebook_name)
PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")

[13:41:08] INFO - Project directory: /mnt/storage1/workspace/lhofer/parnet--globalclip-head


In [3]:
# ── GPU setup ──────────────────────────────────────────────────────────────────
pylbsr.misc.set_seed(42)

params_gpu_index = 0
device = torch.device(f"cuda:{params_gpu_index}")
torch.cuda.set_device(params_gpu_index)

if torch.cuda.is_available():
    logger.info(f"Using GPU: {torch.cuda.get_device_name(device)}")
else:
    logger.warning("No GPU available, using CPU instead.")

[14:10:37] INFO - Using GPU: NVIDIA RTX A4000


Seed set to 42
⏱ 0.15 s (00:00:00)


In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
# Run to load the model from
EVAL_RUN = "parnet.7m-0.0.unfreeze_last_2_layers.globalclip-lysate-noNHS.filtered.noctrl"

# Integrated Gradients settings
params_n_steps: int = 50        # number of interpolation steps
params_n_sequences: int = None   # number of test sequences to compute IG on

# Data
params_seq_length: int = 600
params_num_tasks: int = 1

In [ ]:
# ── Filepaths ──────────────────────────────────────────────────────────────────
_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.yaml").read_text())

FILEPATHS = DotMap()
FILEPATHS["data"] = Path(_fp_cfg["global_clip"]["data_lysate_noNHS_filtered"])
FILEPATHS["pretrained_model"] = Path(_fp_cfg["models"][ParnetModelName.PARNET_7M_0_0.value])
FILEPATHS["run_dir"] = PROJECT_DIR / "results" / "globalclip-head" / "training" / EVAL_RUN
FILEPATHS["output_dir"] = FILEPATHS["run_dir"] / "integrated_gradients"
FILEPATHS["figures_dir"] = PROJECT_DIR / "results" / "globalclip-head" / "figures"

FILEPATHS["output_dir"].mkdir(parents=True, exist_ok=True)
FILEPATHS["figures_dir"].mkdir(parents=True, exist_ok=True)

logger.info(f"Run dir:    {FILEPATHS.run_dir}")
logger.info(f"Output dir: {FILEPATHS.output_dir}")

In [ ]:
# ── Load test dataset ──────────────────────────────────────────────────────────
test_ds = GzListDataset(
    FILEPATHS.data, split="test",
    length=params_seq_length, total_key="globalCLIP", control_key="control",
    shuffle=False, mmap=True,
)

n_sequences = len(test_ds) if params_n_sequences is None else params_n_sequences
logger.info(f"Test size: {len(test_ds)} — computing IG on {n_sequences} sequences")

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────────
class GlobalCLIPHead(torch.nn.Module):
    def __init__(self, num_tasks=1):
        super().__init__()
        self.projection = LinearProjectionHead(num_tasks)

    def forward(self, inputs, **kwargs):
        logit = self.projection(inputs)
        logprob = logit - torch.logsumexp(logit, dim=-1, keepdim=True)
        return {
            'total': logprob,
            'mix_coeff': torch.ones(inputs.shape[0], 1, device=inputs.device),
        }

model = load_parnet_model(
    ParnetModelName.PARNET_7M_0_0,
    FILEPATHS["pretrained_model"],
    dtype=torch.float32,
    device=device,
)

model.head = GlobalCLIPHead(num_tasks=params_num_tasks).to(device)

# initialize lazy parameters
with torch.no_grad():
    _dummy = torch.zeros(1, 512, params_seq_length, device=device)
    model.head(_dummy)

# load fine-tuned weights
_checkpoint = torch.load(FILEPATHS["run_dir"] / "model.statedict.pt", map_location=device)
model.load_state_dict(_checkpoint["state_dict"])
model.eval()
logger.info(f"Model loaded from: {FILEPATHS.run_dir / 'model.statedict.pt'}")

In [ ]:
# ── Integrated Gradients implementation ────────────────────────────────────────
def integrated_gradients(model, sequence, n_steps=50):
    """
    Compute Integrated Gradients for a single sequence.
    
    Args:
        model: fine-tuned GlobalCLIP model
        sequence: (1, 4, 600) one-hot encoded sequence tensor
        n_steps: number of interpolation steps between baseline and input
    
    Returns:
        attributions: (4, 600) attribution scores
    """
    baseline = torch.zeros_like(sequence)  # all-zeros baseline (no information)
    
    # interpolate between baseline and input: (n_steps, 1, 4, 600)
    alphas = torch.linspace(0, 1, n_steps, device=sequence.device)
    interpolated = torch.stack([
        baseline + alpha * (sequence - baseline)
        for alpha in alphas
    ])  # (n_steps, 1, 4, 600)
    interpolated = interpolated.squeeze(1)  # (n_steps, 4, 600)
    interpolated.requires_grad_(True)
    
    # forward pass on all interpolated sequences at once
    output = model(sequence=interpolated)
    target = output['total'].sum()  # scalar
    
    # compute gradients
    target.backward()
    grads = interpolated.grad  # (n_steps, 4, 600)
    
    # average gradients and scale by (input - baseline)
    avg_grads = grads.mean(0)  # (4, 600)
    attributions = avg_grads * (sequence.squeeze(0) - baseline.squeeze(0))
    
    return attributions.detach().cpu()

In [ ]:
# ── Check sequence format from GzListDataset ───────────────────────────────────
sample = test_ds[0]
print(type(sample["inputs"]["sequence"]))
print(sample["inputs"]["sequence"].shape if hasattr(sample["inputs"]["sequence"], "shape") else sample["inputs"]["sequence"][:50])

In [ ]:
# ── Compute IG for all test sequences ──────────────────────────────────────────
all_attributions = []
all_sequences = []
all_metadata = []

model.eval()
for i in tqdm(range(n_sequences)):
    sample = test_ds[i]
    
    # get one-hot encoded sequence (4, 600)
    sequence = sample["inputs"]["sequence"].unsqueeze(0).to(device)  # (1, 4, 600)
    
    # compute IG
    attrs = integrated_gradients(model, sequence, n_steps=params_n_steps)  # (4, 600)
    
    all_attributions.append(attrs.numpy())
    all_sequences.append(sample["inputs"]["sequence"].numpy())
    all_metadata.append(sample["meta"])

all_attributions = np.stack(all_attributions)  # (n_sequences, 4, 600)
all_sequences = np.stack(all_sequences)         # (n_sequences, 4, 600)

logger.info(f"Computed IG for {n_sequences} sequences")
logger.info(f"Attribution maps shape: {all_attributions.shape}")